In [2]:
# -*- coding: utf-8 -*-
"""
Quick patch scan for missing instrumentation evidence in a set of repos.

- Input: Patch.csv (with columns like github_url|repo_url|html_url OR owner,repo)
- Clone default branch (depth=1), init submodules (depth=1)
- Find & copy:
  * TEST CODE -->  ...\RQ1\Patch\Test_Code
      - src/**androidTest**/*.kt|*.java
      - integration_test/**/*.dart, test_driver/**/*.dart
  * COMPLEMENTARY CONFIG --> ...\RQ1\Patch\Config
      - **/*.gradle, **/*.gradle.kts, gradle.properties, settings.gradle(.kts)
      - gradle/libs.versions.toml
      - buildSrc/**/*.{kt,kts,gradle,gradle.kts,properties}
      - gradle/plugins/**/*.{kt,kts,gradle,gradle.kts,properties}
      - any in-repo includeBuild("...") directories (same file globs)

- Flat naming & collision rules match your main collector:
  {owner}.{project}__{ci_token}++{file_lower}[__N].ext

- Output:
  - Files copied into the two target folders
  - Patch_Review_Summary.csv summarizing what was found per repo
"""

from __future__ import annotations
import csv
import os
import re
import shutil
import stat
import subprocess
from pathlib import Path
from typing import Optional, Set, Tuple

import pandas as pd

# ====== CONFIG: edit these if needed ======
PATCH_CSV = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\Patch\Patch.csv"  # or the full path to your Patch.csv
BASE_SAVE_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\Patch")
TEST_SAVE_DIR = BASE_SAVE_DIR / "Test_Code"
CFG_SAVE_DIR  = BASE_SAVE_DIR / "Config"
CLONE_WORKDIR = BASE_SAVE_DIR / "Clones"

SUMMARY_CSV = BASE_SAVE_DIR / "Patch_Review_Summary.csv"

# ====== Helpers ======
def run(cmd: list[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess[str]:
    return subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, check=check)

def sanitize_token(s: str) -> str:
    return re.sub(r"[^a-z0-9._+-]", "_", str(s).lower())

def split_stem_and_suffixes(name: str) -> tuple[str, str]:
    p = Path(name)
    suffixes = ''.join(p.suffixes)
    if suffixes:
        return name[:-len(suffixes)], suffixes
    return name, ""

def _counter_candidate(base_name: str, n: int) -> str:
    if n == 1:
        return base_name
    if "++" in base_name:
        head, tail = base_name.split("++", 1)
        tail_stem, tail_suf = split_stem_and_suffixes(tail)
        return f"{head}++{tail_stem}__{n}{tail_suf}"
    stem, suf = split_stem_and_suffixes(base_name)
    return f"{stem}__{n}{suf}"

def resolve_counter_name(bucket_dir: Path, base_name: str, in_memory_taken: Set[str]) -> str:
    n = 1
    while True:
        candidate = _counter_candidate(base_name, n)
        if candidate not in in_memory_taken and not (bucket_dir / candidate).exists():
            in_memory_taken.add(candidate)
            return candidate
        n += 1

def make_flat_filename(owner: str, project: str, token: str, repo_rel_name: str, used_names: Set[str]) -> str:
    owner_tok   = sanitize_token(owner)
    project_tok = sanitize_token(project)
    token_tok   = sanitize_token(token or "other")
    file_lower  = repo_rel_name.lower()
    base = f"{owner_tok}.{project_tok}__{token_tok}++{file_lower}"
    return resolve_counter_name(bucket_dir=None, base_name=base, in_memory_taken=used_names)  # we return unique name; later we still check FS

def save_with_rule(dest_dir: Path, flat_name: str, src: Path, used_names: Set[str]) -> Path:
    dest_dir.mkdir(parents=True, exist_ok=True)
    # ensure we don't collide on-disk (in case of restarts)
    final_name = resolve_counter_name(dest_dir, flat_name, used_names)
    dest = dest_dir / final_name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dest)
    return dest

def get_default_branch(repo_url: str) -> str:
    # Try ls-remote symref
    try:
        out = run(["git", "ls-remote", "--symref", repo_url, "HEAD"]).stdout
        for line in out.splitlines():
            s = line.strip()
            if s.startswith("ref: ") and s.endswith("HEAD"):
                ref = s.split()[1]
                if ref.startswith("refs/heads/"):
                    return ref.split("/", 2)[2]
    except Exception:
        pass
    # Fallback guesses
    for guess in ("main", "master"):
        try:
            run(["git", "ls-remote", repo_url, f"refs/heads/{guess}"], check=True)
            return guess
        except Exception:
            continue
    raise RuntimeError("Could not determine default branch")

def parse_patch_csv(path: Path) -> list[tuple[str, str]]:
    """
    Returns a list of (owner, repo) pairs from Patch.csv.
    Accepts columns: github_url|repo_url|html_url OR owner & repo.
    """
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.lower()
    urls = None
    for c in ("github_url", "repo_url", "html_url", "url"):
        if c in df.columns:
            urls = df[c].dropna().astype(str).str.strip().tolist()
            break
    out = []
    if urls:
        for u in urls:
            m = re.search(r"github\.com/([^/\s]+)/([^/\s]+)", u)
            if m:
                owner = m.group(1)
                repo  = m.group(2).replace(".git", "")
                out.append((owner, repo))
    else:
        if "owner" in df.columns and "repo" in df.columns:
            for _, r in df[["owner","repo"]].dropna().iterrows():
                out.append((str(r["owner"]).strip(), str(r["repo"]).strip()))
    return out

# patterns
TEST_SRC_RE   = re.compile(r"(^|/|\\)src(/|\\).+androidtest(/|\\).+\.(kt|java)$", re.I)
FLUTTER_IT_RE = re.compile(r"(^|/|\\)(integration_test|test_driver)(/|\\).+\.dart$", re.I)

GRADLE_FILE_RE   = re.compile(r"\.(gradle|gradle\.kts)$", re.I)
SETTINGS_NAME_RE = re.compile(r"settings\.gradle(\.kts)?$", re.I)
GRADLE_PROPS_RE  = re.compile(r"(^|/|\\)gradle\.properties$", re.I)
LIBS_VERS_RE     = re.compile(r"(^|/|\\)gradle(/|\\)libs\.versions\.toml$", re.I)

BUILD_SRC_RE     = re.compile(r"(^|/|\\)buildsrc(/|\\)", re.I)
GRADLE_PLUGINS_RE= re.compile(r"(^|/|\\)gradle(/|\\)plugins(/|\\)", re.I)

INCLUDEBUILD_RE  = re.compile(r'includeBuild\s*\(\s*["\']([^"\']+)["\']\s*\)', re.I)

def collect_include_build_paths(repo_root: Path) -> list[Path]:
    paths = []
    for p in repo_root.rglob("settings.gradle*"):
        try:
            txt = p.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue
        for m in INCLUDEBUILD_RE.finditer(txt):
            rel = m.group(1).strip()
            if rel:
                target = (p.parent / rel).resolve()
                # only in-repo targets (avoid external)
                try:
                    if str(target).startswith(str(repo_root.resolve())) and target.exists():
                        paths.append(target)
                except Exception:
                    pass
    # de-dup while preserving order
    seen = set()
    uniq = []
    for t in paths:
        if t not in seen:
            seen.add(t); uniq.append(t)
    return uniq

def is_test_file(path: Path) -> tuple[bool, str]:
    unix = path.as_posix()
    if TEST_SRC_RE.search(unix):
        return True, ("androidtest_kotlin" if path.suffix.lower()==".kt" else "androidtest_java")
    if FLUTTER_IT_RE.search(unix):
        return True, "androidtest_dart"
    return False, ""

def is_config_candidate(path: Path) -> tuple[bool, str]:
    unix = path.as_posix()
    name = path.name
    if GRADLE_FILE_RE.search(name):
        return True, "gradle"
    if SETTINGS_NAME_RE.search(name):
        return True, "gradle"
    if GRADLE_PROPS_RE.search(unix):
        return True, "gradle_props"
    if LIBS_VERS_RE.search(unix):
        return True, "libs_versions"
    if BUILD_SRC_RE.search(unix):
        if name.lower().endswith((".kt",".kts",".gradle",".gradle.kts")) or name.lower()=="gradle.properties":
            return True, "buildsrc_plugin"
    if GRADLE_PLUGINS_RE.search(unix):
        if name.lower().endswith((".kt",".kts",".gradle",".gradle.kts")) or name.lower()=="gradle.properties":
            return True, "gradle_convention"
    return False, ""

def force_writable_remove(path: Path):
    def onerror(func, p, _):
        try:
            os.chmod(p, stat.S_IWRITE)
            func(p)
        except Exception:
            pass
    shutil.rmtree(path, onerror=onerror)

def main():
    BASE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    TEST_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    CFG_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    CLONE_WORKDIR.mkdir(parents=True, exist_ok=True)

    repos = parse_patch_csv(Path(PATCH_CSV))
    if not repos:
        raise SystemExit("No repos found in Patch.csv (expected github_url/repo_url/html_url OR owner,repo).")

    used_test_names: Set[str] = set()
    used_cfg_names:  Set[str] = set()

    summary_rows = []

    for idx, (owner, repo) in enumerate(repos, 1):
        repo_url = f"https://github.com/{owner}/{repo}.git"
        print(f"\n[{idx}/{len(repos)}] {owner}/{repo}")

        # clone shallow default branch
        try:
            default_branch = get_default_branch(repo_url)
            dest = CLONE_WORKDIR / f"{sanitize_token(owner)}.{sanitize_token(repo)}"
            if dest.exists():
                force_writable_remove(dest)
            run(["git", "clone", "--depth", "1", "--single-branch", "--branch", default_branch, repo_url, str(dest)])
        except Exception as e:
            print(f"  ! clone failed: {e}")
            summary_rows.append({
                "owner": owner, "repo": repo, "default_branch": "", "status": "clone_failed",
                "test_files_copied": 0, "config_files_copied": 0, "notes": str(e)
            })
            continue

        # submodules (depth 1)
        try:
            run(["git", "submodule", "update", "--init", "--depth", "1", "--recursive"], cwd=dest, check=False)
        except Exception as e:
            print(f"  ! submodule init error (ignored): {e}")

        test_count = 0
        cfg_count  = 0
        notes = []

        # includeBuild paths
        include_paths = collect_include_build_paths(dest)
        if include_paths:
            notes.append(f"includeBuild_paths={len(include_paths)}")

        # walk main repo
        candidates = [dest] + include_paths
        for root in candidates:
            for p in root.rglob("*"):
                if not p.is_file():
                    continue
                rel = p.relative_to(root).as_posix()  # repo-relative within this root
                try:
                    is_test, token = is_test_file(p)
                    if is_test:
                        flat = f"{sanitize_token(owner)}.{sanitize_token(repo)}__{token}++{rel.lower()}"
                        saved = save_with_rule(TEST_SAVE_DIR, flat, p, used_test_names)
                        test_count += 1
                        continue

                    is_cfg, token = is_config_candidate(p)
                    if is_cfg:
                        flat = f"{sanitize_token(owner)}.{sanitize_token(repo)}__{token}++{rel.lower()}"
                        saved = save_with_rule(CFG_SAVE_DIR, flat, p, used_cfg_names)
                        cfg_count += 1
                        continue
                except Exception as e:
                    print(f"  ! copy error {rel}: {e}")

        summary_rows.append({
            "owner": owner,
            "repo": repo,
            "default_branch": default_branch,
            "status": "ok",
            "test_files_copied": test_count,
            "config_files_copied": cfg_count,
            "notes": "; ".join(notes) if notes else ""
        })

        # cleanup clone to keep disk tidy
        try:
            force_writable_remove(dest)
        except Exception:
            pass

    # write summary
    pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")
    print(f"\nDone. Summary saved to: {SUMMARY_CSV}")
    print(f"Test files saved to:    {TEST_SAVE_DIR}")
    print(f"Config files saved to:  {CFG_SAVE_DIR}")

if __name__ == "__main__":
    main()



[1/36] TimotheeJeannin/ProviGen

[2/36] kevinhinterlong/archwiki-viewer

[3/36] chaosbastler/opentraining

[4/36] jakenjarvis/Android-OrmLiteContentProvider

[5/36] wuyexiong/transparent-over-animtabsview

[6/36] OpnTec/bodyapps-android

[7/36] Stuart-campbell/RushOrm

[8/36] kost/NetworkMapper

[9/36] SilenceIM/Silence

[10/36] Redgram/redgram-for-reddit

[11/36] whilu/AndroidTagView

[12/36] XuDaojie/MultiStateView

[13/36] TUBB/CalendarSelector

[14/36] karonl/InDoorSurfaceView

[15/36] streetcomplete/StreetComplete

[16/36] BennyKok/PxerStudio

[17/36] PureWriter/about-page

[18/36] drakeet/Floo

[19/36] harryjph/android-bluetooth-serial

[20/36] Link184/KidAdapter

[21/36] Sharkaboi/MediaHub

[22/36] ericktijerou/jetpuppy

[23/36] fornewid/material-motion-compose

[24/36] Mercandj/android-dev-challenge-compose-4

[25/36] overpas/compose-treemap-chart

[26/36] wireapp/kalium

[27/36] deepmedia/Knee

[28/36] xxfast/Decompose-Router

[29/36] msasikanth/twine

[30/36] mbakgun/midjour